<a href="https://colab.research.google.com/github/haniyathul/TESIS-HANIY/blob/main/Pemodelan_Data_Empiris_KKP_572_573.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import zipfile

daftar_zip = [
    "/content/HASIL_PEMBAGIAN_CV.zip",
    "/content/HASIL_LENGKAP_ESF_TUNA.zip"
]

for path in daftar_zip:
    print("\nFILE:", os.path.basename(path))

    if os.path.exists(path):
        with zipfile.ZipFile(path, "r") as z:
            isi = z.namelist()
            print("✅ ZIP ditemukan")
            print("Jumlah file:", len(isi))
            print("Contoh isi:")
            for nama in isi[:10]:
                print(" -", nama)
    else:
        print("❌ ZIP belum ditemukan")



FILE: HASIL_PEMBAGIAN_CV.zip
✅ ZIP ditemukan
Jumlah file: 5
Contoh isi:
 - REKAP_UKURAN_FOLD.csv
 - METADATA_CV.json
 - INDEKS_TRAIN_VALIDASI.csv
 - IDENTITAS_FOLD_401_DATA.csv
 - PETA_RANDOM_DAN_SPATIAL_CV.png

FILE: HASIL_LENGKAP_ESF_TUNA.zip
✅ ZIP ditemukan
Jumlah file: 133
Contoh isi:
 - 01_PETA/GRID_LENGKAP_WPP_572_573.csv
 - 01_PETA/HEATMAP_KORELASI_XY.png
 - 01_PETA/PETA_10_VARIABEL_OSEANOGRAFI.png
 - 01_PETA/PETA_CPUE_GRID_INDONESIA.png
 - 01_PETA/PETA_GRID_LENGKAP_WPP_572_573.png
 - 02_EDA/BOXPLOT_CPUE_DAN_X.png
 - 02_EDA/HISTOGRAM_CPUE_DAN_X.png
 - 02_EDA/KORELASI_PEARSON_SPEARMAN.png
 - 02_EDA/KORELASI_X_TERHADAP_CPUE.png
 - 02_EDA/TABEL_KORELASI_XY.csv


In [2]:

# ==========================================================
# PEMULIHAN DATA PENELITIAN DARI 2 FILE ZIP
#
# 1. HASIL_LENGKAP_ESF_TUNA.zip
# 2. HASIL_PEMBAGIAN_CV.zip
#
# OUTPUT:
# - DATASET_MODEL : 13 dataset
# - CV_SPLITS     : Random 10-Fold + Spatial 4-Fold
#
# Tidak mengulang EDA, W, eigenvector, atau seleksi.
# ==========================================================

import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from io import BytesIO

# ----------------------------------------------------------
# 1. LOKASI ZIP
# ----------------------------------------------------------

ZIP_DATA = Path(
    "/content/HASIL_LENGKAP_ESF_TUNA.zip"
)

ZIP_CV = Path(
    "/content/HASIL_PEMBAGIAN_CV.zip"
)

for path in [ZIP_DATA, ZIP_CV]:
    if not path.exists():
        raise FileNotFoundError(
            f"Belum ditemukan: {path.name}. "
            "Tunggu upload selesai terlebih dahulu."
        )

# ----------------------------------------------------------
# 2. DAFTAR 13 DATASET
# ----------------------------------------------------------

KODE_DATASET = [
    "BASELINE",
    "MORAN_KNN5",
    "MORAN_DISTANCE",
    "MORAN_IDW",
    "LASSO_KNN5",
    "LASSO_DISTANCE",
    "LASSO_IDW",
    "STEPWISE_KNN5",
    "STEPWISE_DISTANCE",
    "STEPWISE_IDW",
    "SHAP_KNN5",
    "SHAP_DISTANCE",
    "SHAP_IDW"
]

DATASET_MODEL = {}

with zipfile.ZipFile(ZIP_DATA, "r") as z:

    semua_file = z.namelist()

    for kode in KODE_DATASET:

        nama_dicari = f"{kode}.csv"

        # Cari file di folder mana pun dalam ZIP
        kandidat = [
            f for f in semua_file
            if Path(f).name == nama_dicari
            and "09_DATASET_13_MODEL" in f
        ]

        if not kandidat:
            # Cadangan jika struktur folder berbeda
            kandidat = [
                f for f in semua_file
                if Path(f).name == nama_dicari
            ]

        if not kandidat:
            raise FileNotFoundError(
                f"{nama_dicari} tidak ditemukan "
                "dalam ZIP hasil penelitian."
            )

        with z.open(kandidat[0]) as f:
            DATASET_MODEL[kode] = pd.read_csv(f)

print("✅ Berhasil memuat", len(DATASET_MODEL), "dataset")

# ----------------------------------------------------------
# 3. CEK KESEJAJARAN DATASET
# ----------------------------------------------------------

BASE = DATASET_MODEL["BASELINE"]
N = len(BASE)

for kode, data in DATASET_MODEL.items():

    if len(data) != N:
        raise ValueError(
            f"Jumlah baris berbeda: {kode}"
        )

    pd.testing.assert_frame_equal(
        data[["Longitude", "Latitude", "cpue"]],
        BASE[["Longitude", "Latitude", "cpue"]]
    )

print("✅ Semua dataset memiliki", N, "observasi")

# ----------------------------------------------------------
# 4. BACA INDEKS CV ASLI DARI ZIP
# ----------------------------------------------------------

with zipfile.ZipFile(ZIP_CV, "r") as z:

    kandidat = [
        f for f in z.namelist()
        if Path(f).name == "INDEKS_TRAIN_VALIDASI.csv"
    ]

    if not kandidat:
        raise FileNotFoundError(
            "INDEKS_TRAIN_VALIDASI.csv "
            "tidak ditemukan dalam ZIP CV."
        )

    with z.open(kandidat[0]) as f:
        rincian_cv = pd.read_csv(f)

# ----------------------------------------------------------
# 5. REKONSTRUKSI SPLIT
# ----------------------------------------------------------

CV_SPLITS = {}

KONFIG_CV = {
    "RANDOM_10FOLD": 10,
    "SPATIAL_4FOLD": 4
}

for jenis_cv, jumlah_fold in KONFIG_CV.items():

    bagian_cv = rincian_cv[
        rincian_cv["Jenis CV"] == jenis_cv
    ]

    daftar_split = []

    for fold in range(1, jumlah_fold + 1):

        bagian_fold = bagian_cv[
            bagian_cv["Fold"] == fold
        ]

        # ID_Observasi pada file sebelumnya
        # berbasis 1, sedangkan iloc berbasis 0.
        idx_train = (
            bagian_fold[
                bagian_fold["Peran"] == "TRAIN"
            ]["ID_Observasi"]
            .to_numpy(dtype=int) - 1
        )

        idx_valid = (
            bagian_fold[
                bagian_fold["Peran"] == "VALIDASI"
            ]["ID_Observasi"]
            .to_numpy(dtype=int) - 1
        )

        # Validasi indeks
        assert len(idx_train) > 0
        assert len(idx_valid) > 0

        assert set(idx_train).isdisjoint(
            set(idx_valid)
        )

        assert len(
            np.union1d(idx_train, idx_valid)
        ) == N

        daftar_split.append(
            (idx_train, idx_valid)
        )

    CV_SPLITS[jenis_cv] = daftar_split

# ----------------------------------------------------------
# 6. VALIDASI SEMUA OBSERVASI MUNCUL SEKALI
# ----------------------------------------------------------

for jenis_cv, daftar_split in CV_SPLITS.items():

    seluruh_validasi = np.concatenate([
        idx_valid
        for _, idx_valid in daftar_split
    ])

    assert len(seluruh_validasi) == N

    assert np.array_equal(
        np.sort(seluruh_validasi),
        np.arange(N)
    )

# ----------------------------------------------------------
# 7. TAMPILKAN REKAP
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("PEMULIHAN DATA DAN CV BERHASIL")
print("=" * 60)

print("Jumlah dataset        :", len(DATASET_MODEL))
print("Jumlah observasi      :", N)
print(
    "Random CV             :",
    len(CV_SPLITS["RANDOM_10FOLD"]),
    "fold"
)
print(
    "Spatial CV            :",
    len(CV_SPLITS["SPATIAL_4FOLD"]),
    "fold"
)

print("\n✅ DATASET_MODEL tersedia")
print("✅ CV_SPLITS tersedia")
print("✅ Fold asli berhasil dipulihkan")
print("✅ Siap menjalankan baseline 5 trial")


✅ Berhasil memuat 13 dataset
✅ Semua dataset memiliki 401 observasi

PEMULIHAN DATA DAN CV BERHASIL
Jumlah dataset        : 13
Jumlah observasi      : 401
Random CV             : 10 fold
Spatial CV            : 4 fold

✅ DATASET_MODEL tersedia
✅ CV_SPLITS tersedia
✅ Fold asli berhasil dipulihkan
✅ Siap menjalankan baseline 5 trial


In [ ]:

# ================================================================
# PEMODELAN OTOMATIS GW-XGBOOST TWEEDIE
#
# 1 BASELINE + 12 MODEL ESF = 13 MODEL
#
# OPTIMASI RINGAN:
#   3 KOMBINASI HYPERPARAMETER TERENCANA
#   Optuna GridSampler
#
# OUTER CV:
#   Random 10-Fold
#   Spatial Cluster 4-Fold
#
# INNER CV:
#   3-Fold
#   Seluruh target validasi inner digunakan
#
# BOBOT:
#   Adaptive Gaussian + Haversine
#
# CHECKPOINT:
#   Google Drive setiap trial dan setiap fold
#
# OUTPUT:
#   RMSE, MAE, R2
#   Prediksi out-of-fold
#   Parameter terbaik
#   Grafik perbandingan 13 model
#   CSV dan ZIP
# ================================================================

!pip -q install optuna xgboost

import json
import shutil
import sqlite3
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from pathlib import Path
from google.colab import drive
from IPython.display import display
from xgboost import XGBRegressor

from sklearn.model_selection import KFold, GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import haversine_distances
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

# ================================================================
# 1. PENGATURAN
# ================================================================

SEED = 2026
R_BUMI = 6371.0088
N_INNER = 3

# Tiga konfigurasi yang akan dicoba.
KONFIGURASI = [
    {
        "n_estimators": 50,
        "bandwidth_k": 45,
        "tweedie_variance_power": 1.3
    },
    {
        "n_estimators": 75,
        "bandwidth_k": 60,
        "tweedie_variance_power": 1.5
    },
    {
        "n_estimators": 100,
        "bandwidth_k": 90,
        "tweedie_variance_power": 1.7
    }
]

# Parameter XGBoost lainnya tetap.
PARAM_TETAP = {
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.3,
    "reg_alpha": 0.3,
    "reg_lambda": 2.0,
    "min_child_weight": 7
}

VAR_X = [
    "so", "thetao", "zos", "uo", "vo",
    "chl", "o2", "mlotst", "nppv",
    "Kecepatan"
]

METODE = [
    "MORAN", "LASSO",
    "STEPWISE", "SHAP"
]

MATRIKS_W = [
    "KNN5", "DISTANCE", "IDW"
]

DAFTAR_MODEL = ["BASELINE"] + [
    f"{m}_{w}"
    for m in METODE
    for w in MATRIKS_W
]

DAFTAR_CV = [
    "RANDOM_10FOLD",
    "SPATIAL_4FOLD"
]

# ================================================================
# 2. GOOGLE DRIVE
# ================================================================

drive.mount("/content/drive")

FOLDER = Path(
    "/content/drive/MyDrive/"
    "HASIL_13_MODEL_3KOMBINASI"
)

FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

FOLDER_DB = Path(
    "/content/DB_13_MODEL_3KOMBINASI"
)

FOLDER_DB.mkdir(
    parents=True,
    exist_ok=True
)

# ================================================================
# 3. VALIDASI INPUT
# ================================================================

if "DATASET_MODEL" not in globals():
    raise NameError(
        "DATASET_MODEL belum tersedia. "
        "Jalankan cell pemulihan ZIP dahulu."
    )

if "CV_SPLITS" not in globals():
    raise NameError(
        "CV_SPLITS belum tersedia. "
        "Jalankan cell pemulihan ZIP dahulu."
    )

BASE = DATASET_MODEL["BASELINE"]
N = len(BASE)

if len(CV_SPLITS["RANDOM_10FOLD"]) != 10:
    raise ValueError("Random CV bukan 10 fold.")

if len(CV_SPLITS["SPATIAL_4FOLD"]) != 4:
    raise ValueError("Spatial CV bukan 4 fold.")

# Periksa bahwa semua titik mendapat tepat
# satu prediksi validasi pada tiap jenis CV.
for jenis in DAFTAR_CV:

    seluruh_validasi = []

    for it, iv in CV_SPLITS[jenis]:

        it = np.asarray(it, dtype=int)
        iv = np.asarray(iv, dtype=int)

        assert len(np.intersect1d(it, iv)) == 0

        assert np.array_equal(
            np.union1d(it, iv),
            np.arange(N)
        )

        seluruh_validasi.extend(iv.tolist())

    assert np.array_equal(
        np.sort(seluruh_validasi),
        np.arange(N)
    )

DATA_SIAP = {}

for kode in DAFTAR_MODEL:

    if kode not in DATASET_MODEL:
        raise KeyError(
            f"Dataset tidak ditemukan: {kode}"
        )

    df = DATASET_MODEL[kode].copy()

    if len(df) != N:
        raise ValueError(
            f"Jumlah baris berbeda: {kode}"
        )

    pd.testing.assert_frame_equal(
        df[["Longitude", "Latitude", "cpue"]],
        BASE[["Longitude", "Latitude", "cpue"]]
    )

    fitur = [
        c for c in df.columns
        if c not in [
            "Longitude", "Latitude", "cpue"
        ]
    ]

    if not set(VAR_X).issubset(fitur):
        raise ValueError(
            f"Prediktor dasar tidak lengkap: {kode}"
        )

    X = df[fitur].to_numpy(dtype=float)
    y = df["cpue"].to_numpy(dtype=float)

    koordinat = df[
        ["Latitude", "Longitude"]
    ].to_numpy(dtype=float)

    if not (
        np.isfinite(X).all()
        and np.isfinite(y).all()
        and np.isfinite(koordinat).all()
    ):
        raise ValueError(
            f"Data tidak valid: {kode}"
        )

    if np.any(y < 0):
        raise ValueError(
            f"CPUE negatif: {kode}"
        )

    DATA_SIAP[kode] = {
        "X": X,
        "y": y,
        "coord": koordinat,
        "fitur": fitur
    }

print("=" * 65)
print("PEMODELAN 13 MODEL GW-XGBOOST TWEEDIE")
print("=" * 65)

print("Observasi       :", N)
print("Jumlah model    :", len(DAFTAR_MODEL))
print("Kombinasi tuning:", len(KONFIGURASI))
print("Inner CV        :", N_INNER)
print("Random CV       : 10 fold")
print("Spatial CV      : 4 fold")
print("Folder Drive    :", FOLDER)

print("\nTIGA KOMBINASI YANG DICOBA:")
display(
    pd.DataFrame(KONFIGURASI)
)

# ================================================================
# 4. IDENTITAS EKSPERIMEN
# ================================================================

def identitas_eksperimen(kode, jenis, it, iv):

    d = DATA_SIAP[kode]

    h = hashlib.sha256()

    for arr in [
        d["X"], d["y"], d["coord"]
    ]:
        h.update(
            np.ascontiguousarray(
                arr,
                dtype=np.float64
            ).tobytes()
        )

    h.update(
        np.asarray(it, dtype=np.int64).tobytes()
    )

    h.update(
        np.asarray(iv, dtype=np.int64).tobytes()
    )

    h.update(
        json.dumps({
            "kode": kode,
            "jenis": jenis,
            "inner": N_INNER,
            "seed": SEED,
            "kombinasi": KONFIGURASI,
            "tetap": PARAM_TETAP
        }, sort_keys=True).encode()
    )

    return h.hexdigest()[:20]


# ================================================================
# 5. JARAK HAVERSINE
# ================================================================

def jarak_km(target, train):

    return haversine_distances(
        np.radians(target),
        np.radians(train)
    ) * R_BUMI


# ================================================================
# 6. BOBOT ADAPTIVE GAUSSIAN
# ================================================================

def adaptive_gaussian(d, k):

    d = np.asarray(d, dtype=float)

    if not (1 <= k <= len(d)):
        raise ValueError(
            f"bandwidth_k={k} tidak valid "
            f"untuk {len(d)} data latih."
        )

    b = float(
        np.partition(d, k - 1)[k - 1]
    )

    b = max(b, 1e-6)

    w = np.exp(
        -0.5 * (d / b) ** 2
    )

    rata = float(w.mean())

    if not np.isfinite(rata) or rata <= 0:
        raise ValueError(
            "Bobot geografis tidak valid."
        )

    return w / rata, b


# ================================================================
# 7. PREDIKSI GW-XGBOOST LOKAL
# ================================================================

def prediksi_lokal(
    X_train,
    y_train,
    X_target,
    D_target_train,
    konfigurasi
):

    params = {
        **PARAM_TETAP,
        **konfigurasi
    }

    k = int(params.pop("bandwidth_k"))

    params.update({
        "objective": "reg:tweedie",
        "tree_method": "hist",
        "random_state": SEED,
        "n_jobs": 1,
        "verbosity": 0
    })

    hasil = np.empty(len(X_target))
    bandwidth = np.empty(len(X_target))

    for i in range(len(X_target)):

        w, b = adaptive_gaussian(
            D_target_train[i], k
        )

        model = XGBRegressor(**params)

        model.fit(
            X_train,
            y_train,
            sample_weight=w
        )

        hasil[i] = float(
            model.predict(
                X_target[i:i+1]
            )[0]
        )

        bandwidth[i] = b

    return hasil, bandwidth


# ================================================================
# 8. INNER CV - TETAP 3 FOLD
# ================================================================

def buat_inner(
    X,
    y,
    koordinat,
    jenis_cv
):

    if jenis_cv == "RANDOM_10FOLD":

        cv = KFold(
            n_splits=N_INNER,
            shuffle=True,
            random_state=SEED
        )

        raw = list(cv.split(X))

    elif jenis_cv == "SPATIAL_4FOLD":

        lon = koordinat[:, 1]
        lat = koordinat[:, 0]

        lat_ref = np.deg2rad(
            np.mean(lat)
        )

        XY = np.column_stack([
            lon * 111.32 * np.cos(lat_ref),
            lat * 111.32
        ])

        kelompok = KMeans(
            n_clusters=N_INNER,
            n_init=20,
            random_state=SEED
        ).fit_predict(XY)

        cv = GroupKFold(
            n_splits=N_INNER
        )

        raw = list(
            cv.split(
                X, y, groups=kelompok
            )
        )

    else:
        raise ValueError("Jenis CV tidak dikenali.")

    return [
        {
            "train": it,
            "valid": iv,
            "D": jarak_km(
                koordinat[iv],
                koordinat[it]
            )
        }
        for it, iv in raw
    ]


# ================================================================
# 9. BACKUP DATABASE OPTUNA
# ================================================================

def backup_database(db_lokal, db_drive):

    with sqlite3.connect(str(db_lokal)) as src:
        with sqlite3.connect(str(db_drive)) as dst:
            src.backup(dst)


# ================================================================
# 10. TUNING 3 KOMBINASI DENGAN OPTUNA GRID
# ================================================================

def tuning_tiga_kombinasi(
    kode,
    jenis_cv,
    fold_ke,
    X,
    y,
    koordinat,
    id_exp
):

    inner = buat_inner(
        X, y, koordinat, jenis_cv
    )

    min_train = min(
        len(b["train"])
        for b in inner
    )

    k_terbesar = max(
        c["bandwidth_k"]
        for c in KONFIGURASI
    )

    if min_train < k_terbesar:
        raise ValueError(
            f"{kode} {jenis_cv} Fold {fold_ke}: "
            f"train inner terkecil {min_train}, "
            f"sedangkan bandwidth_k maksimum "
            f"{k_terbesar}. "
            "Perlu menyesuaikan rancangan "
            "bandwidth sebelum melanjutkan."
        )

    nama_db = (
        f"OPTUNA_{kode}_{jenis_cv}_"
        f"FOLD{fold_ke}.db"
    )

    db_lokal = FOLDER_DB / nama_db
    db_drive = FOLDER / nama_db

    # Pulihkan database dari Drive bila runtime baru.
    if db_drive.exists() and not db_lokal.exists():
        shutil.copy2(db_drive, db_lokal)

    # Untuk menghindari tertukar dengan eksperimen
    # lama, nama studi memuat identitas eksperimen.
    nama_studi = (
        f"GRID3_{kode}_{jenis_cv}_"
        f"FOLD{fold_ke}_{id_exp}"
    )

    sampler = optuna.samplers.GridSampler({
        "config_id": [0, 1, 2]
    })

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        study_name=nama_studi,
        storage=f"sqlite:///{db_lokal}",
        load_if_exists=True
    )

    def objective(trial):

        config_id = trial.suggest_categorical(
            "config_id",
            [0, 1, 2]
        )

        konfigurasi = KONFIGURASI[config_id]

        aktual = []
        prediksi = []

        for bagian in inner:

            it = bagian["train"]
            iv = bagian["valid"]

            pred, _ = prediksi_lokal(
                X[it],
                y[it],
                X[iv],
                bagian["D"],
                konfigurasi
            )

            if not np.isfinite(pred).all():
                raise ValueError(
                    "Prediksi inner tidak valid."
                )

            aktual.extend(y[iv].tolist())
            prediksi.extend(pred.tolist())

        return float(
            np.sqrt(
                mean_squared_error(
                    aktual,
                    prediksi
                )
            )
        )

    def callback(study, trial):

        if trial.state == optuna.trial.TrialState.COMPLETE:

            no_konfig = trial.params["config_id"] + 1

            print(
                f"    Kombinasi {no_konfig}/3 | "
                f"RMSE inner = {trial.value:.5f}",
                flush=True
            )

        backup_database(
            db_lokal,
            db_drive
        )

        study.trials_dataframe().to_csv(
            FOLDER /
            f"RIWAYAT_{kode}_{jenis_cv}_"
            f"FOLD{fold_ke}.csv",
            index=False
        )

    # Evaluasi hanya kombinasi yang belum selesai.
    selesai_id = {
        int(t.params["config_id"])
        for t in study.trials
        if (
            t.state == optuna.trial.TrialState.COMPLETE
            and "config_id" in t.params
        )
    }

    sisa = len(
        set([0, 1, 2]) - selesai_id
    )

    if sisa > 0:

        study.optimize(
            objective,
            n_trials=sisa,
            callbacks=[callback],
            gc_after_trial=True,
            show_progress_bar=False
        )

    backup_database(db_lokal, db_drive)

    # Pastikan semua konfigurasi selesai.
    selesai_id = {
        int(t.params["config_id"])
        for t in study.trials
        if (
            t.state == optuna.trial.TrialState.COMPLETE
            and "config_id" in t.params
        )
    }

    if selesai_id != {0, 1, 2}:
        raise RuntimeError(
            "Belum semua 3 konfigurasi selesai."
        )

    terbaik_id = int(
        study.best_params["config_id"]
    )

    return (
        KONFIGURASI[terbaik_id].copy(),
        float(study.best_value)
    )


# ================================================================
# 11. LOOP OTOMATIS 13 MODEL × 2 CV
# ================================================================

print("\nMEMULAI PEMODELAN")
print("13 model × (10 Random + 4 Spatial)")
print("3 konfigurasi per outer fold")
print("Semua hasil disimpan di Google Drive.")

for nomor, kode in enumerate(
    DAFTAR_MODEL,
    start=1
):

    print("\n" + "#" * 65)
    print(f"MODEL {nomor}/13 : {kode}")
    print("#" * 65)

    data = DATA_SIAP[kode]

    X_all = data["X"]
    y_all = data["y"]
    coord_all = data["coord"]

    for jenis_cv in DAFTAR_CV:

        for fold_ke, (it, iv) in enumerate(
            CV_SPLITS[jenis_cv],
            start=1
        ):

            it = np.asarray(it, dtype=int)
            iv = np.asarray(iv, dtype=int)

            prefix = (
                f"{kode}_{jenis_cv}_FOLD{fold_ke}"
            )

            path_hasil = (
                FOLDER / f"HASIL_{prefix}.csv"
            )

            path_pred = (
                FOLDER / f"PREDIKSI_{prefix}.csv"
            )

            path_param = (
                FOLDER / f"PARAM_{prefix}.json"
            )

            id_exp = identitas_eksperimen(
                kode, jenis_cv, it, iv
            )

            print(
                f"\n{kode} | {jenis_cv} | "
                f"Fold {fold_ke}/{len(CV_SPLITS[jenis_cv])}",
                flush=True
            )

            # ------------------------------------
            # CHECKPOINT
            # ------------------------------------

            sudah_selesai = False

            if (
                path_hasil.exists()
                and path_pred.exists()
                and path_param.exists()
            ):

                try:

                    with open(path_param) as f:
                        record = json.load(f)

                    pred_lama = pd.read_csv(path_pred)

                    sudah_selesai = (
                        record.get("ID_Exp") == id_exp
                        and len(pred_lama) == len(iv)
                        and set(
                            pred_lama["ID_Observasi"]
                            .astype(int)
                        ) == set(iv + 1)
                    )

                except Exception:
                    sudah_selesai = False

            if sudah_selesai:

                print("✅ Sudah selesai, dilewati.")
                continue

            # ------------------------------------
            # DATA OUTER
            # ------------------------------------

            X_train = X_all[it]
            y_train = y_all[it]
            coord_train = coord_all[it]

            X_valid = X_all[iv]
            y_valid = y_all[iv]
            coord_valid = coord_all[iv]

            # ------------------------------------
            # TUNING 3 KOMBINASI
            # ------------------------------------

            params_terbaik, rmse_inner = (
                tuning_tiga_kombinasi(
                    kode,
                    jenis_cv,
                    fold_ke,
                    X_train,
                    y_train,
                    coord_train,
                    id_exp
                )
            )

            # ------------------------------------
            # PREDIKSI VALIDASI OUTER
            # ------------------------------------

            D_outer = jarak_km(
                coord_valid,
                coord_train
            )

            pred, bandwidth = prediksi_lokal(
                X_train,
                y_train,
                X_valid,
                D_outer,
                params_terbaik
            )

            if not np.isfinite(pred).all():
                raise ValueError(
                    f"Prediksi tidak valid: {prefix}"
                )

            # ------------------------------------
            # METRIK FOLD
            # ------------------------------------

            rmse = float(
                np.sqrt(
                    mean_squared_error(
                        y_valid, pred
                    )
                )
            )

            mae = float(
                mean_absolute_error(
                    y_valid, pred
                )
            )

            r2 = float(
                r2_score(
                    y_valid, pred
                )
            )

            hasil = pd.DataFrame([{
                "Model": kode,
                "Jenis_CV": jenis_cv,
                "Fold": fold_ke,
                "N_Train": len(it),
                "N_Validasi": len(iv),
                "Jumlah_Fitur": X_train.shape[1],
                "Jumlah_EV": X_train.shape[1] - 10,
                "RMSE_Inner": rmse_inner,
                "RMSE": rmse,
                "MAE": mae,
                "R2": r2,
                "n_estimators": params_terbaik[
                    "n_estimators"
                ],
                "bandwidth_k": params_terbaik[
                    "bandwidth_k"
                ],
                "tweedie_variance_power": params_terbaik[
                    "tweedie_variance_power"
                ]
            }])

            pred_df = pd.DataFrame({
                "Model": kode,
                "Jenis_CV": jenis_cv,
                "Fold": fold_ke,
                "ID_Observasi": iv + 1,
                "Longitude": coord_valid[:, 1],
                "Latitude": coord_valid[:, 0],
                "CPUE_Aktual": y_valid,
                "CPUE_Prediksi": pred,
                "Residual": y_valid - pred,
                "Bandwidth_KM": bandwidth
            })

            record_param = {
                "Model": kode,
                "Jenis_CV": jenis_cv,
                "Fold": fold_ke,
                "ID_Exp": id_exp,
                "Jumlah_Kombinasi": 3,
                "RMSE_Inner": rmse_inner,
                "Parameter_Terbaik": params_terbaik,
                "Parameter_Tetap": PARAM_TETAP
            }

            # Simpan file hasil terakhir sebagai
            # penanda fold berhasil diselesaikan.
            with open(path_param, "w") as f:
                json.dump(record_param, f, indent=2)

            pred_df.to_csv(
                path_pred,
                index=False
            )

            hasil.to_csv(
                path_hasil,
                index=False
            )

            print(
                f"✅ Fold selesai | "
                f"RMSE={rmse:.4f} | "
                f"MAE={mae:.4f} | "
                f"R2={r2:.4f}",
                flush=True
            )

# ================================================================
# 12. REKAP SEMUA FOLD DAN PREDIKSI
# ================================================================

semua_hasil = []
semua_prediksi = []

for kode in DAFTAR_MODEL:

    for jenis_cv in DAFTAR_CV:

        for fold_ke in range(
            1,
            len(CV_SPLITS[jenis_cv]) + 1
        ):

            prefix = (
                f"{kode}_{jenis_cv}_FOLD{fold_ke}"
            )

            path_hasil = (
                FOLDER / f"HASIL_{prefix}.csv"
            )

            path_pred = (
                FOLDER / f"PREDIKSI_{prefix}.csv"
            )

            if not (
                path_hasil.exists()
                and path_pred.exists()
            ):
                raise FileNotFoundError(
                    f"Belum selesai: {prefix}"
                )

            semua_hasil.append(
                pd.read_csv(path_hasil)
            )

            semua_prediksi.append(
                pd.read_csv(path_pred)
            )

REKAP_SEMUA_FOLD = pd.concat(
    semua_hasil,
    ignore_index=True
)

PREDIKSI_OOF_13_MODEL = pd.concat(
    semua_prediksi,
    ignore_index=True
)

REKAP_SEMUA_FOLD.to_csv(
    FOLDER / "REKAP_SELURUH_FOLD.csv",
    index=False
)

PREDIKSI_OOF_13_MODEL.to_csv(
    FOLDER / "PREDIKSI_OOF_13_MODEL.csv",
    index=False
)

# ================================================================
# 13. HITUNG RMSE, MAE, R2 PER MODEL DAN CV
# ================================================================

rekap = []

for kode in DAFTAR_MODEL:

    for jenis_cv in DAFTAR_CV:

        bagian = PREDIKSI_OOF_13_MODEL[
            (PREDIKSI_OOF_13_MODEL["Model"] == kode)
            &
            (PREDIKSI_OOF_13_MODEL["Jenis_CV"] == jenis_cv)
        ].sort_values("ID_Observasi")

        if not np.array_equal(
            bagian["ID_Observasi"].to_numpy(dtype=int),
            np.arange(1, N + 1)
        ):
            raise ValueError(
                f"OOF tidak lengkap: {kode}, {jenis_cv}"
            )

        y = bagian["CPUE_Aktual"].to_numpy()
        pred = bagian["CPUE_Prediksi"].to_numpy()

        rekap.append({
            "Model": kode,
            "Metode_CV": jenis_cv,
            "Jumlah_Fold": len(CV_SPLITS[jenis_cv]),
            "N_Observasi": N,
            "Jumlah_EV": len(
                DATA_SIAP[kode]["fitur"]
            ) - 10,
            "RMSE": float(
                np.sqrt(
                    mean_squared_error(y, pred)
                )
            ),
            "MAE": float(
                mean_absolute_error(y, pred)
            ),
            "R2": float(r2_score(y, pred))
        })

PERBANDINGAN_13_MODEL = pd.DataFrame(rekap)

print("\n" + "=" * 65)
print("PERBANDINGAN 13 MODEL")
print("=" * 65)

display(
    PERBANDINGAN_13_MODEL.round(5)
)

PERBANDINGAN_13_MODEL.to_csv(
    FOLDER / "PERBANDINGAN_13_MODEL.csv",
    index=False
)

# ================================================================
# 14. GRAFIK PERBANDINGAN METRIK
# ================================================================

for metrik in ["RMSE", "MAE", "R2"]:

    pivot = PERBANDINGAN_13_MODEL.pivot(
        index="Model",
        columns="Metode_CV",
        values=metrik
    ).reindex(DAFTAR_MODEL)

    fig, ax = plt.subplots(
        figsize=(15, 6),
        dpi=140
    )

    pivot.plot(
        kind="bar",
        ax=ax,
        color=["#3182bd", "#e67e22"],
        width=0.8
    )

    ax.axhline(
        0, color="black", linewidth=0.8
    )

    ax.set_title(
        f"Perbandingan {metrik} - 13 Model\n"
        "Random 10-Fold vs Spatial 4-Fold"
    )

    ax.set_xlabel("Model")
    ax.set_ylabel(metrik)
    ax.grid(axis="y", alpha=0.2)
    ax.legend(title="Metode CV")

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    plt.savefig(
        FOLDER / f"GRAFIK_{metrik}_13_MODEL.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# ================================================================
# 15. GRAFIK AKTUAL VS PREDIKSI BASELINE
# ================================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5.5),
    dpi=140
)

for ax, jenis_cv in zip(axes, DAFTAR_CV):

    bagian = PREDIKSI_OOF_13_MODEL[
        (PREDIKSI_OOF_13_MODEL["Model"] == "BASELINE")
        &
        (PREDIKSI_OOF_13_MODEL["Jenis_CV"] == jenis_cv)
    ]

    y = bagian["CPUE_Aktual"].to_numpy()
    pred = bagian["CPUE_Prediksi"].to_numpy()

    batas = max(
        float(y.max()),
        float(pred.max())
    )

    ax.scatter(
        y, pred,
        color="#3182bd",
        s=25,
        alpha=0.65
    )

    ax.plot(
        [0, batas],
        [0, batas],
        "--",
        color="#d95f02"
    )

    ax.set_title(jenis_cv)
    ax.set_xlabel("CPUE Aktual")
    ax.set_ylabel("CPUE Prediksi")
    ax.grid(alpha=0.2)

fig.suptitle(
    "Baseline GW-XGBoost Tweedie\n"
    "Aktual vs Prediksi Out-of-Fold"
)

plt.tight_layout()

plt.savefig(
    FOLDER / "GRAFIK_AKTUAL_VS_PREDIKSI_BASELINE.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 16. METADATA
# ================================================================

metadata = {
    "Model": "Baseline + 12 ESF-GW-XGBoost Tweedie",
    "Jumlah_Model": 13,
    "Jumlah_Observasi": N,
    "Jumlah_Prediktor_Dasar": len(VAR_X),
    "Optimasi": "Optuna GridSampler",
    "Jumlah_Konfigurasi": 3,
    "Konfigurasi": KONFIGURASI,
    "Parameter_Tetap": PARAM_TETAP,
    "Inner_CV": N_INNER,
    "Random_Outer": 10,
    "Spatial_Outer": 4,
    "Spatial_Method": "KMeans + GroupKFold",
    "Kernel": "Adaptive Gaussian",
    "Distance": "Haversine km",
    "Seed": SEED,
    "Catatan": (
        "Seleksi LASSO, Stepwise, dan SHAP "
        "dilakukan sebelum CV menggunakan seluruh "
        "respons. Evaluasi ESF bersifat eksploratif "
        "karena berpotensi response leakage."
    )
}

with open(
    FOLDER / "METADATA_PENELITIAN.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)

# ================================================================
# 17. BUAT ZIP HASIL
# ================================================================

PATH_ZIP = shutil.make_archive(
    "/content/HASIL_13_MODEL_3KOMBINASI",
    "zip",
    str(FOLDER)
)

ZIP_DRIVE = (
    FOLDER.parent /
    "HASIL_13_MODEL_3KOMBINASI.zip"
)

shutil.copy2(
    PATH_ZIP,
    ZIP_DRIVE
)

print("\n" + "=" * 65)
print("PEMODELAN SELESAI")
print("=" * 65)

print("✅ Baseline + 12 model ESF")
print("✅ Random 10-Fold")
print("✅ Spatial 4-Fold")
print("✅ Inner 3-Fold")
print("✅ Tiga kombinasi hyperparameter diuji")
print("✅ Hasil per fold tersimpan di Drive")
print("✅ Prediksi out-of-fold lengkap")
print("✅ Grafik perbandingan tersedia")
print("✅ ZIP tersimpan di Google Drive")

print("\nFolder:", FOLDER)
print("ZIP:", ZIP_DRIVE)


Mounted at /content/drive
PEMODELAN 13 MODEL GW-XGBOOST TWEEDIE
Observasi       : 401
Jumlah model    : 13
Kombinasi tuning: 3
Inner CV        : 3
Random CV       : 10 fold
Spatial CV      : 4 fold
Folder Drive    : /content/drive/MyDrive/HASIL_13_MODEL_3KOMBINASI

TIGA KOMBINASI YANG DICOBA:


,n_estimators,bandwidth_k,tweedie_variance_power
0,50,45,1.3
1,75,60,1.5
2,100,90,1.7



MEMULAI PEMODELAN
13 model × (10 Random + 4 Spatial)
3 konfigurasi per outer fold
Semua hasil disimpan di Google Drive.

#################################################################
MODEL 1/13 : BASELINE
#################################################################

BASELINE | RANDOM_10FOLD | Fold 1/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 2/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 3/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 4/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 5/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 6/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 7/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 8/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 9/10
✅ Sudah selesai, dilewati.

BASELINE | RANDOM_10FOLD | Fold 10/10
✅ Sudah selesai, dilewati.

BASELINE | SPATIAL_4FOLD | Fold 1/4
✅ Sudah selesai, dilewati.

BASELINE

In [ ]:

# ================================================================
# CELL 1 - EVALUASI AKHIR 13 MODEL
# ================================================================
# INPUT : DATASET_MODEL dan hasil per fold di Google Drive
# OUTPUT:
#   PERBANDINGAN_13_MODEL
#   PILIHAN_5_MODEL
#   MODEL_UNGGULAN
#   MORAN_GLOBAL_HASIL
#   Grafik RMSE, MAE, R2, selisih CV, perbaikan baseline
#   Grafik jumlah EV dan Moran's I residual
# ================================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics.pairwise import haversine_distances
from IPython.display import display

SEED = 2026
N_PERMUTASI = 999

FOLDER_HASIL = Path(
    "/content/drive/MyDrive/HASIL_13_MODEL_3KOMBINASI"
)
FOLDER_ANALISIS = Path(
    "/content/drive/MyDrive/ANALISIS_AKHIR_5_MODEL"
)
FOLDER_ANALISIS.mkdir(parents=True, exist_ok=True)

if "DATASET_MODEL" not in globals():
    raise NameError("Pulihkan DATASET_MODEL dari ZIP dahulu.")

MODEL_SEMUA = ["BASELINE"] + [
    f"{metode}_{w}"
    for metode in ["MORAN", "LASSO", "STEPWISE", "SHAP"]
    for w in ["KNN5", "DISTANCE", "IDW"]
]
JENIS_CV = ["RANDOM_10FOLD", "SPATIAL_4FOLD"]
N_FOLD = {"RANDOM_10FOLD": 10, "SPATIAL_4FOLD": 4}
VAR_X = [
    "so", "thetao", "zos", "uo", "vo",
    "chl", "o2", "mlotst", "nppv", "Kecepatan"
]

BASE = DATASET_MODEL["BASELINE"]
N = len(BASE)

# ------------------------------------------------
# 1. PERIKSA SEMUA HASIL OOF
# ------------------------------------------------

semua_prediksi = []
semua_fold = []
belum_selesai = []

for kode in MODEL_SEMUA:
    for cv in JENIS_CV:
        for fold in range(1, N_FOLD[cv] + 1):
            prefix = f"{kode}_{cv}_FOLD{fold}"
            fp = FOLDER_HASIL / f"PREDIKSI_{prefix}.csv"
            fh = FOLDER_HASIL / f"HASIL_{prefix}.csv"

            if not fp.exists() or not fh.exists():
                belum_selesai.append(prefix)
                continue

            p = pd.read_csv(fp)
            h = pd.read_csv(fh)
            p["Model"], p["Jenis_CV"], p["Fold"] = kode, cv, fold
            h["Model"], h["Jenis_CV"], h["Fold"] = kode, cv, fold

            semua_prediksi.append(p)
            semua_fold.append(h)

if belum_selesai:
    print("Contoh fold yang belum lengkap:", belum_selesai[:15])
    raise RuntimeError(
        f"Masih ada {len(belum_selesai)} fold belum lengkap. "
        "Lanjutkan pemodelan terlebih dahulu."
    )

PREDIKSI_OOF_13_MODEL = pd.concat(
    semua_prediksi, ignore_index=True
)
REKAP_FOLD_13_MODEL = pd.concat(
    semua_fold, ignore_index=True
)

# ------------------------------------------------
# 2. METRIK POOLED OOF, JUMLAH EV
# ------------------------------------------------

rekap = []

for kode in MODEL_SEMUA:
    if kode not in DATASET_MODEL:
        raise KeyError(f"Dataset {kode} tidak tersedia.")

    fitur = [
        c for c in DATASET_MODEL[kode].columns
        if c not in ["Longitude", "Latitude", "cpue"]
    ]
    n_ev = len([c for c in fitur if c not in VAR_X])

    for cv in JENIS_CV:
        p = PREDIKSI_OOF_13_MODEL.query(
            "Model == @kode and Jenis_CV == @cv"
        ).sort_values("ID_Observasi")

        ids = p["ID_Observasi"].to_numpy(dtype=int)
        if not np.array_equal(ids, np.arange(1, N + 1)):
            raise ValueError(f"OOF tidak lengkap: {kode}, {cv}")

        y = p["CPUE_Aktual"].to_numpy(float)
        yh = p["CPUE_Prediksi"].to_numpy(float)

        if not np.isfinite(yh).all():
            raise ValueError(f"Prediksi tidak valid: {kode}, {cv}")

        rekap.append({
            "Model": kode,
            "Metode_CV": cv,
            "Jumlah_EV": n_ev,
            "RMSE": np.sqrt(mean_squared_error(y, yh)),
            "MAE": mean_absolute_error(y, yh),
            "R2": r2_score(y, yh)
        })

PERBANDINGAN_13_MODEL = pd.DataFrame(rekap)
PERBANDINGAN_13_MODEL.to_csv(
    FOLDER_ANALISIS / "PERBANDINGAN_13_MODEL.csv",
    index=False
)

print("\nPERBANDINGAN 13 MODEL - 2 METODE CV")
display(PERBANDINGAN_13_MODEL.round(5))

# ------------------------------------------------
# 3. SELISIH RANDOM DAN SPATIAL
# ------------------------------------------------

lebar = PERBANDINGAN_13_MODEL.pivot(
    index="Model",
    columns="Metode_CV",
    values=["RMSE", "MAE", "R2"]
).reindex(MODEL_SEMUA)

SELISIH_CV = pd.DataFrame({
    "Model": MODEL_SEMUA,
    "RMSE_Random": lebar[("RMSE", "RANDOM_10FOLD")].to_numpy(),
    "RMSE_Spatial": lebar[("RMSE", "SPATIAL_4FOLD")].to_numpy(),
    "MAE_Random": lebar[("MAE", "RANDOM_10FOLD")].to_numpy(),
    "MAE_Spatial": lebar[("MAE", "SPATIAL_4FOLD")].to_numpy(),
    "R2_Random": lebar[("R2", "RANDOM_10FOLD")].to_numpy(),
    "R2_Spatial": lebar[("R2", "SPATIAL_4FOLD")].to_numpy()
})

SELISIH_CV["Delta_RMSE_Spatial_minus_Random"] = (
    SELISIH_CV["RMSE_Spatial"] - SELISIH_CV["RMSE_Random"]
)
SELISIH_CV["Delta_MAE_Spatial_minus_Random"] = (
    SELISIH_CV["MAE_Spatial"] - SELISIH_CV["MAE_Random"]
)
SELISIH_CV["Delta_R2_Spatial_minus_Random"] = (
    SELISIH_CV["R2_Spatial"] - SELISIH_CV["R2_Random"]
)

SELISIH_CV.to_csv(
    FOLDER_ANALISIS / "SELISIH_RANDOM_VS_SPATIAL.csv",
    index=False
)

# ------------------------------------------------
# 4. PERSENTASE PERBAIKAN TERHADAP BASELINE
# ------------------------------------------------

perbaikan = []

for cv in JENIS_CV:
    bagian = PERBANDINGAN_13_MODEL.query(
        "Metode_CV == @cv"
    )

    base_rmse = float(
        bagian.loc[bagian["Model"] == "BASELINE", "RMSE"].iloc[0]
    )
    base_mae = float(
        bagian.loc[bagian["Model"] == "BASELINE", "MAE"].iloc[0]
    )

    for _, r in bagian.iterrows():
        perbaikan.append({
            "Model": r["Model"],
            "Metode_CV": cv,
            "Perbaikan_RMSE_pct": (
                100 * (base_rmse - r["RMSE"]) / base_rmse
            ),
            "Perbaikan_MAE_pct": (
                100 * (base_mae - r["MAE"]) / base_mae
            )
        })

PERBAIKAN_BASELINE = pd.DataFrame(perbaikan)
PERBAIKAN_BASELINE.to_csv(
    FOLDER_ANALISIS / "PERBAIKAN_TERHADAP_BASELINE.csv",
    index=False
)

# ------------------------------------------------
# 5. PILIH 5 MODEL AKHIR BERDASARKAN RMSE SPATIAL
# ------------------------------------------------

pilihan = [{
    "Kelompok": "BASELINE",
    "Model": "BASELINE",
    "W": "-",
    "Kriteria": "Acuan"
}]

for metode in ["MORAN", "LASSO", "STEPWISE", "SHAP"]:
    kandidat = PERBANDINGAN_13_MODEL[
        (PERBANDINGAN_13_MODEL["Metode_CV"] == "SPATIAL_4FOLD") &
        PERBANDINGAN_13_MODEL["Model"].str.startswith(metode + "_")
    ].sort_values(["RMSE", "MAE", "Model"])

    kode = kandidat.iloc[0]["Model"]

    pilihan.append({
        "Kelompok": metode,
        "Model": kode,
        "W": kode.split("_", 1)[1],
        "Kriteria": "RMSE Spatial terkecil dalam kelompok"
    })

PILIHAN_5_MODEL = pd.DataFrame(pilihan)

# Gabungkan nilai kedua CV supaya mudah ditafsirkan
PILIHAN_5_MODEL = PILIHAN_5_MODEL.merge(
    SELISIH_CV,
    on="Model",
    how="left"
)

MODEL_UNGGULAN = (
    PILIHAN_5_MODEL.sort_values(
        ["RMSE_Spatial", "MAE_Spatial", "Model"]
    ).iloc[0]["Model"]
)

PILIHAN_5_MODEL["Model_Unggulan"] = (
    PILIHAN_5_MODEL["Model"] == MODEL_UNGGULAN
)

print("\nLIMA MODEL AKHIR:")
display(PILIHAN_5_MODEL.round(5))
print("\nMODEL UNGGULAN KESELURUHAN:", MODEL_UNGGULAN)

PILIHAN_5_MODEL.to_csv(
    FOLDER_ANALISIS / "PILIHAN_5_MODEL.csv",
    index=False
)

# ------------------------------------------------
# 6. JUMLAH EV SETIAP MODEL
# ------------------------------------------------

JUMLAH_EV_ESF = (
    PERBANDINGAN_13_MODEL[
        PERBANDINGAN_13_MODEL["Metode_CV"] == "SPATIAL_4FOLD"
    ][["Model", "Jumlah_EV"]]
    .copy()
)

JUMLAH_EV_ESF.to_csv(
    FOLDER_ANALISIS / "JUMLAH_EV_13_MODEL.csv",
    index=False
)

# ------------------------------------------------
# 7. GRAFIK RMSE, MAE, R2
# ------------------------------------------------

for metrik in ["RMSE", "MAE", "R2"]:
    pivot = PERBANDINGAN_13_MODEL.pivot(
        index="Model",
        columns="Metode_CV",
        values=metrik
    ).reindex(MODEL_SEMUA)

    fig, ax = plt.subplots(figsize=(16, 6), dpi=140)
    x = np.arange(len(MODEL_SEMUA))
    lebar_bar = 0.38

    ax.bar(
        x - lebar_bar / 2,
        pivot["RANDOM_10FOLD"],
        width=lebar_bar,
        color="#3182bd",
        label="Random 10-Fold"
    )
    ax.bar(
        x + lebar_bar / 2,
        pivot["SPATIAL_4FOLD"],
        width=lebar_bar,
        color="#e67e22",
        label="Spatial 4-Fold"
    )

    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x, MODEL_SEMUA, rotation=50, ha="right")
    ax.set_ylabel(metrik)
    ax.set_title(f"Perbandingan {metrik} - 13 Model")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()

    plt.savefig(
        FOLDER_ANALISIS / f"BARPLOT_{metrik}_13_MODEL.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()

# ------------------------------------------------
# 8. GRAFIK DELTA RMSE SPATIAL - RANDOM
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5), dpi=140)
ax.bar(
    SELISIH_CV["Model"],
    SELISIH_CV["Delta_RMSE_Spatial_minus_Random"],
    color="#9467bd"
)
ax.axhline(0, color="black", lw=0.9)
ax.set_ylabel("RMSE Spatial - RMSE Random")
ax.set_title("Perubahan RMSE antara Dua Metode CV")
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=50, ha="right")
plt.tight_layout()
plt.savefig(
    FOLDER_ANALISIS / "BARPLOT_DELTA_RMSE_CV.png",
    dpi=300, bbox_inches="tight"
)
plt.show()

# ------------------------------------------------
# 9. GRAFIK PERBAIKAN RMSE TERHADAP BASELINE
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 5), dpi=140)

piv = PERBAIKAN_BASELINE.pivot(
    index="Model",
    columns="Metode_CV",
    values="Perbaikan_RMSE_pct"
).reindex(MODEL_SEMUA)

x = np.arange(len(MODEL_SEMUA))
ax.bar(
    x - 0.19, piv["RANDOM_10FOLD"], 0.38,
    color="#3182bd", label="Random"
)
ax.bar(
    x + 0.19, piv["SPATIAL_4FOLD"], 0.38,
    color="#e67e22", label="Spatial"
)
ax.axhline(0, color="black", lw=0.9)
ax.set_xticks(x, MODEL_SEMUA, rotation=50, ha="right")
ax.set_ylabel("Perbaikan RMSE terhadap baseline (%)")
ax.set_title("Perbaikan atau Penurunan Performa Model ESF")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig(
    FOLDER_ANALISIS / "BARPLOT_PERBAIKAN_RMSE.png",
    dpi=300, bbox_inches="tight"
)
plt.show()

# ------------------------------------------------
# 10. GRAFIK JUMLAH EV
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 5), dpi=140)
bars = ax.bar(
    JUMLAH_EV_ESF["Model"],
    JUMLAH_EV_ESF["Jumlah_EV"],
    color="#4c78a8"
)
ax.bar_label(bars, padding=2, fontsize=8)
ax.set_title("Jumlah Eigenvector yang Digunakan")
ax.set_ylabel("Jumlah EV")
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=50, ha="right")
plt.tight_layout()
plt.savefig(
    FOLDER_ANALISIS / "BARPLOT_JUMLAH_EV.png",
    dpi=300, bbox_inches="tight"
)
plt.show()

# ------------------------------------------------
# 11. MORAN'S I GLOBAL UNTUK CPUE DAN RESIDUAL OOF
# W sesuai definisi semula:
# KNN5 simetris; Distance 122.33 km;
# IDW radius 203.88 km, bobot 1/d
# ------------------------------------------------

D = haversine_distances(
    np.radians(
        BASE[["Latitude", "Longitude"]].to_numpy(float)
    )
) * 6371.0088

np.fill_diagonal(D, np.inf)

W_STORE_DIAG = {}

# KNN-5 simetris
tetangga = np.argpartition(D, kth=4, axis=1)[:, :5]
W_knn = np.zeros((N, N), dtype=float)
W_knn[np.arange(N)[:, None], tetangga] = 1.0
W_knn = np.maximum(W_knn, W_knn.T)
np.fill_diagonal(W_knn, 0)
W_STORE_DIAG["KNN5"] = csr_matrix(W_knn)

# Distance Band
W_db = (D <= 122.33).astype(float)
np.fill_diagonal(W_db, 0)
W_STORE_DIAG["DISTANCE"] = csr_matrix(W_db)

# IDW
with np.errstate(divide="ignore", invalid="ignore"):
    W_idw = np.where(
        (D <= 203.88) & (D > 0), 1 / D, 0.0
    )
np.fill_diagonal(W_idw, 0)
W_STORE_DIAG["IDW"] = csr_matrix(W_idw)

def hitung_moran(y, W, n_perm=999, seed=2026):
    y = np.asarray(y, dtype=float)
    z = y - y.mean()
    s0 = float(W.sum())
    denominator = float(z @ z)

    if s0 <= 0 or denominator <= 0:
        return np.nan, np.nan

    I = float(
        len(y) / s0 * (z @ (W @ z)) / denominator
    )

    rng = np.random.default_rng(seed)
    simulasi = np.empty(n_perm)

    for j in range(n_perm):
        zp = rng.permutation(z)
        simulasi[j] = (
            len(y) / s0 * (zp @ (W @ zp)) / denominator
        )

    I0 = -1 / (len(y) - 1)

    p = (
        1 + np.count_nonzero(
            np.abs(simulasi - I0) >= abs(I - I0)
        )
    ) / (n_perm + 1)

    return I, p

data_moran = [{
    "Model": "CPUE_ASLI",
    "Jenis_CV": "-",
    "Nilai": BASE["cpue"].to_numpy(float)
}]

for kode in PILIHAN_5_MODEL["Model"]:
    for cv in JENIS_CV:
        p = PREDIKSI_OOF_13_MODEL.query(
            "Model == @kode and Jenis_CV == @cv"
        ).sort_values("ID_Observasi")

        residual = (
            p["CPUE_Aktual"].to_numpy(float) -
            p["CPUE_Prediksi"].to_numpy(float)
        )

        data_moran.append({
            "Model": kode,
            "Jenis_CV": cv,
            "Nilai": residual
        })

rekap_moran = []

for item in data_moran:
    for nama_w, W in W_STORE_DIAG.items():
        I, p = hitung_moran(item["Nilai"], W, N_PERMUTASI)

        rekap_moran.append({
            "Model": item["Model"],
            "Jenis_CV": item["Jenis_CV"],
            "Matriks_W": nama_w,
            "Morans_I": I,
            "P_Value": p,
            "Signifikan_5pct": (
                bool(p < 0.05) if np.isfinite(p) else None
            )
        })

MORAN_GLOBAL_HASIL = pd.DataFrame(rekap_moran)

print("\nMORAN'S I GLOBAL:")
display(MORAN_GLOBAL_HASIL.round(5))

MORAN_GLOBAL_HASIL.to_csv(
    FOLDER_ANALISIS / "MORAN_GLOBAL_CPUE_RESIDUAL.csv",
    index=False
)

# ------------------------------------------------
# 12. BARPLOT MORAN RESIDUAL
# ------------------------------------------------

for cv in JENIS_CV:
    m = MORAN_GLOBAL_HASIL.query(
        "Jenis_CV == @cv"
    )

    piv = m.pivot(
        index="Model", columns="Matriks_W", values="Morans_I"
    ).reindex(PILIHAN_5_MODEL["Model"])

    ax = piv.plot(
        kind="bar",
        figsize=(11, 5),
        color=["#4c78a8", "#f28e2b", "#59a14f"],
        width=0.8
    )
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Moran's I Residual OOF - {cv}")
    ax.set_ylabel("Moran's I")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.2)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(
        FOLDER_ANALISIS / f"BARPLOT_MORAN_RESIDUAL_{cv}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()

# ------------------------------------------------
# 13. SIMPAN OBJEK PENTING
# ------------------------------------------------

PREDIKSI_OOF_13_MODEL.to_csv(
    FOLDER_ANALISIS / "PREDIKSI_OOF_13_MODEL.csv",
    index=False
)
REKAP_FOLD_13_MODEL.to_csv(
    FOLDER_ANALISIS / "REKAP_FOLD_13_MODEL.csv",
    index=False
)

print("\n✅ CELL 1 SELESAI")
print("✅ Perbandingan dua metode CV tersedia")
print("✅ 5 model akhir dan 1 model unggulan dipilih")
print("✅ Grafik RMSE, MAE, R², delta CV, perbaikan baseline")
print("✅ Jumlah EV dan Moran's I residual tersedia")
print("✅ Semua hasil disimpan ke Google Drive")


In [ ]:

# ================================================================
# CELL 2 - SHAP DAN FEATURE IMPORTANCE 5 MODEL AKHIR
# ================================================================
# SHAP dihitung dari model XGBoost lokal berbobot
# Adaptive Gaussian pada 40 lokasi contoh.
#
# OUTPUT:
#   PARAM_FINAL_DICT
#   FEATURE_IMPORTANCE_5_MODEL
#   SHAP_LOKAL_5_MODEL
#   Grafik SHAP Top 15 per model
#   Grafik perbandingan 10 X
#   Grafik kontribusi kelompok X vs EV
# ================================================================

!pip -q install xgboost

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from xgboost import XGBRegressor
from sklearn.metrics.pairwise import haversine_distances
from pathlib import Path
from IPython.display import display

SEED = 2026
N_SAMPEL_SHAP = 40
R_BUMI = 6371.0088

FOLDER_HASIL = Path(
    "/content/drive/MyDrive/HASIL_13_MODEL_3KOMBINASI"
)
FOLDER_ANALISIS = Path(
    "/content/drive/MyDrive/ANALISIS_AKHIR_5_MODEL"
)
FOLDER_ANALISIS.mkdir(parents=True, exist_ok=True)

VAR_X = [
    "so", "thetao", "zos", "uo", "vo",
    "chl", "o2", "mlotst", "nppv", "Kecepatan"
]

PARAM_TETAP = {
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.3,
    "reg_alpha": 0.3,
    "reg_lambda": 2.0,
    "min_child_weight": 7
}

KONFIGURASI = [
    {"n_estimators": 50, "bandwidth_k": 45,
     "tweedie_variance_power": 1.3},
    {"n_estimators": 75, "bandwidth_k": 60,
     "tweedie_variance_power": 1.5},
    {"n_estimators": 100, "bandwidth_k": 90,
     "tweedie_variance_power": 1.7}
]

if "DATASET_MODEL" not in globals():
    raise NameError("Pulihkan DATASET_MODEL dahulu.")

if "PILIHAN_5_MODEL" not in globals():
    PILIHAN_5_MODEL = pd.read_csv(
        FOLDER_ANALISIS / "PILIHAN_5_MODEL.csv"
    )

DAFTAR_5 = PILIHAN_5_MODEL["Model"].tolist()

# ------------------------------------------------
# 1. TETAPKAN PARAMETER FINAL SECARA TRANSPARAN
#
# Ambil konfigurasi yang paling sering terpilih
# pada 4 outer fold Spatial.
# Jika seri, pilih mean RMSE inner terendah.
#
# Ini adalah aturan finalisasi praktis; bukan
# tuning ulang dengan seluruh 401 data.
# ------------------------------------------------

PARAM_FINAL_DICT = {}
rekap_param = []

for kode in DAFTAR_5:

    daftar = []

    for fold in range(1, 5):
        path = FOLDER_HASIL / (
            f"PARAM_{kode}_SPATIAL_4FOLD_FOLD{fold}.json"
        )

        if not path.exists():
            raise FileNotFoundError(
                f"Parameter Spatial belum ditemukan: {path.name}"
            )

        with open(path, "r") as f:
            catatan = json.load(f)

        p = catatan["Parameter_Terbaik"]

        daftar.append({
            "n_estimators": int(p["n_estimators"]),
            "bandwidth_k": int(p["bandwidth_k"]),
            "tweedie_variance_power": float(
                p["tweedie_variance_power"]
            ),
            "RMSE_Inner": float(catatan["RMSE_Inner"])
        })

    dfp = pd.DataFrame(daftar)

    ringkas = (
        dfp.groupby(
            ["n_estimators", "bandwidth_k",
             "tweedie_variance_power"],
            as_index=False
        )
        .agg(
            Frekuensi=("RMSE_Inner", "size"),
            Rata_Rata_RMSE_Inner=("RMSE_Inner", "mean")
        )
        .sort_values(
            ["Frekuensi", "Rata_Rata_RMSE_Inner"],
            ascending=[False, True]
        )
    )

    terbaik = ringkas.iloc[0]

    p_final = {
        "n_estimators": int(terbaik["n_estimators"]),
        "bandwidth_k": int(terbaik["bandwidth_k"]),
        "tweedie_variance_power": float(
            terbaik["tweedie_variance_power"]
        )
    }

    PARAM_FINAL_DICT[kode] = p_final

    rekap_param.append({
        "Model": kode,
        **p_final,
        "Frekuensi_4_Fold_Spatial": int(
            terbaik["Frekuensi"]
        )
    })

PARAM_FINAL_5_MODEL = pd.DataFrame(rekap_param)

print("PARAMETER FINAL LIMA MODEL:")
display(PARAM_FINAL_5_MODEL)

PARAM_FINAL_5_MODEL.to_csv(
    FOLDER_ANALISIS / "PARAM_FINAL_5_MODEL.csv",
    index=False
)

# ------------------------------------------------
# 2. PILIH 40 LOKASI YANG SAMA
# ------------------------------------------------

N = len(DATASET_MODEL["BASELINE"])
rng = np.random.default_rng(SEED)

IDX_SHAP = np.sort(
    rng.choice(
        np.arange(N),
        size=min(N_SAMPEL_SHAP, N),
        replace=False
    )
)

pd.DataFrame({
    "ID_Observasi": IDX_SHAP + 1
}).to_csv(
    FOLDER_ANALISIS / "SAMPEL_LOKASI_SHAP.csv",
    index=False
)

# ------------------------------------------------
# 3. SHAP LOKAL SETIAP MODEL
# Checkpoint CSV per model untuk menghindari
# pelatihan ulang jika Colab terputus.
# ------------------------------------------------

SEMUA_IMPORTANCE = []
SEMUA_SHAP = []

for kode in DAFTAR_5:

    print("\nSHAP LOKAL:", kode)

    path_shap = (
        FOLDER_ANALISIS / f"SHAP_LOKAL_{kode}.csv"
    )

    df = DATASET_MODEL[kode]
    fitur = [
        c for c in df.columns
        if c not in ["Longitude", "Latitude", "cpue"]
    ]

    X_df = df[fitur].copy()
    y = df["cpue"].to_numpy(float)

    coords = df[
        ["Latitude", "Longitude"]
    ].to_numpy(float)

    konfigurasi = PARAM_FINAL_DICT[kode]

    # Validasi checkpoint berdasarkan ID dan fitur
    reuse = False

    if path_shap.exists():
        lama = pd.read_csv(path_shap)

        reuse = (
            lama["ID_Observasi"].tolist() ==
            (IDX_SHAP + 1).tolist()
            and all(c in lama.columns for c in fitur)
        )

    if reuse:
        shap_df = lama.copy()
        print("✅ SHAP tersimpan ditemukan, dilewati.")

    else:
        dmat = haversine_distances(
            np.radians(coords[IDX_SHAP]),
            np.radians(coords)
        ) * R_BUMI

        p_model = {
            **PARAM_TETAP,
            "n_estimators": konfigurasi["n_estimators"],
            "tweedie_variance_power":
                konfigurasi["tweedie_variance_power"],
            "objective": "reg:tweedie",
            "tree_method": "hist",
            "random_state": SEED,
            "n_jobs": 1,
            "verbosity": 0
        }

        k = konfigurasi["bandwidth_k"]
        shap_rows = []

        for nomor, idx in enumerate(IDX_SHAP):
            d = dmat[nomor].copy()

            # Bandwidth dari tetangga selain diri sendiri
            d_bw = d.copy()
            d_bw[idx] = np.inf

            b = max(
                float(np.partition(d_bw, k - 1)[k - 1]),
                1e-6
            )

            bobot = np.exp(-0.5 * (d / b) ** 2)
            bobot /= bobot.mean()

            model = XGBRegressor(**p_model)
            model.fit(
                X_df,
                y,
                sample_weight=bobot
            )

            dm_target = xgb.DMatrix(
                X_df.iloc[[idx]],
                feature_names=fitur
            )

            kontribusi = model.get_booster().predict(
                dm_target,
                pred_contribs=True
            )

            shap_rows.append(kontribusi[0, :-1])

            if (nomor + 1) % 10 == 0:
                print(
                    f"  {nomor + 1}/{len(IDX_SHAP)} lokasi"
                )

        shap_df = pd.DataFrame(
            np.vstack(shap_rows),
            columns=fitur
        )

        shap_df.insert(
            0, "ID_Observasi", IDX_SHAP + 1
        )
        shap_df.to_csv(path_shap, index=False)

    SEMUA_SHAP.append(
        shap_df.assign(Model=kode)
    )

    nilai = shap_df[fitur].to_numpy(float)
    mean_abs = np.abs(nilai).mean(axis=0)

    imp = pd.DataFrame({
        "Model": kode,
        "Fitur": fitur,
        "Mean_Abs_SHAP": mean_abs,
        "Jenis_Fitur": [
            "Oseanografi" if c in VAR_X
            else "EV_ESF"
            for c in fitur
        ]
    }).sort_values("Mean_Abs_SHAP", ascending=False)

    SEMUA_IMPORTANCE.append(imp)

    # ------------------------------------------------
    # 4. TOP 15 FITUR MASING-MASING MODEL
    # ------------------------------------------------

    top = imp.head(15).iloc[::-1]
    warna = [
        "#3182bd" if t == "Oseanografi"
        else "#e67e22"
        for t in top["Jenis_Fitur"]
    ]

    fig, ax = plt.subplots(figsize=(9, 6), dpi=140)
    ax.barh(
        top["Fitur"],
        top["Mean_Abs_SHAP"],
        color=warna
    )
    ax.set_title(f"SHAP Feature Importance - {kode}")
    ax.set_xlabel("Mean |SHAP| (skala margin log)")
    ax.grid(axis="x", alpha=0.2)

    plt.tight_layout()
    plt.savefig(
        FOLDER_ANALISIS / f"SHAP_TOP15_{kode}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

FEATURE_IMPORTANCE_5_MODEL = pd.concat(
    SEMUA_IMPORTANCE,
    ignore_index=True
)
SHAP_LOKAL_5_MODEL = pd.concat(
    SEMUA_SHAP,
    ignore_index=True
)

FEATURE_IMPORTANCE_5_MODEL.to_csv(
    FOLDER_ANALISIS / "FEATURE_IMPORTANCE_5_MODEL.csv",
    index=False
)
SHAP_LOKAL_5_MODEL.to_csv(
    FOLDER_ANALISIS / "SHAP_LOKAL_5_MODEL.csv",
    index=False
)

# ------------------------------------------------
# 5. PERBANDINGAN SHAP 10 VARIABEL X
# ------------------------------------------------

x_imp = FEATURE_IMPORTANCE_5_MODEL[
    FEATURE_IMPORTANCE_5_MODEL["Jenis_Fitur"] ==
    "Oseanografi"
]

pivot = x_imp.pivot(
    index="Fitur",
    columns="Model",
    values="Mean_Abs_SHAP"
).reindex(VAR_X)

ax = pivot.plot(
    kind="bar",
    figsize=(15, 6),
    width=0.82
)
ax.set_title("Perbandingan SHAP Variabel Oseanografi")
ax.set_ylabel("Mean |SHAP|")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    FOLDER_ANALISIS / "SHAP_10_VARIABEL_X.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------
# 6. KONTRIBUSI GRUP X VS EV
# ------------------------------------------------

agregat = (
    FEATURE_IMPORTANCE_5_MODEL
    .groupby(["Model", "Jenis_Fitur"], as_index=False)
    ["Mean_Abs_SHAP"].sum()
)

pivot_agregat = agregat.pivot(
    index="Model",
    columns="Jenis_Fitur",
    values="Mean_Abs_SHAP"
).reindex(DAFTAR_5).fillna(0)

ax = pivot_agregat.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 5)
)
ax.set_title("Agregat SHAP: Oseanografi vs EV ESF")
ax.set_ylabel("Jumlah Mean |SHAP|")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(
    FOLDER_ANALISIS / "SHAP_GRUP_X_VS_EV.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("\n✅ CELL 2 SELESAI")
print("✅ SHAP 5 model dan feature importance tersedia")
print("✅ Parameter final tersimpan")
print("✅ Semua grafik disimpan di Google Drive")


In [ ]:

# ================================================================
# CELL 3 - PREDIKSI GRID ABU-ABU DAN 5 PETA CPUE
# ================================================================
#
# 5 MODEL:
#   BASELINE
#   MORAN TERBAIK
#   LASSO TERBAIK
#   STEPWISE TERBAIK
#   SHAP TERBAIK
#
# INPUT:
#   DATASET_MODEL
#   PILIHAN_5_MODEL
#   PARAM_FINAL_DICT
#   HASIL_LENGKAP_ESF_TUNA.zip
#
# KHUSUS GRID ABU:
#   Wajib memiliki 10 variabel X yang lengkap.
#   Bila belum tersedia, kode membuat template
#   dan berhenti sebelum prediksi.
#
# OUTPUT:
#   5 CSV prediksi
#   5 peta prediksi
#   1 gambar gabungan 5 peta
#   Checkpoint per 25 target
# ================================================================

import os
import re
import json
import hashlib
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import haversine_distances
from xgboost import XGBRegressor
from IPython.display import display

SEED = 2026
R_BUMI_KM = 6371.0088

VAR_X = [
    "so", "thetao", "zos", "uo", "vo",
    "chl", "o2", "mlotst", "nppv", "Kecepatan"
]

PARAM_TETAP = {
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.3,
    "reg_alpha": 0.3,
    "reg_lambda": 2.0,
    "min_child_weight": 7
}

FOLDER_ANALISIS = Path(
    "/content/drive/MyDrive/ANALISIS_AKHIR_5_MODEL"
)
FOLDER_ANALISIS.mkdir(parents=True, exist_ok=True)

ZIP_PENELITIAN = Path(
    "/content/HASIL_LENGKAP_ESF_TUNA.zip"
)

# File ini bisa diupload nanti ke /content.
FILE_GRID_DENGAN_X = Path(
    "/content/GRID_ABU_DENGAN_10_X.csv"
)

if "DATASET_MODEL" not in globals():
    raise NameError("Pulihkan DATASET_MODEL dahulu.")

if "PILIHAN_5_MODEL" not in globals():
    PILIHAN_5_MODEL = pd.read_csv(
        FOLDER_ANALISIS / "PILIHAN_5_MODEL.csv"
    )

if "PARAM_FINAL_DICT" not in globals():
    tabel = pd.read_csv(
        FOLDER_ANALISIS / "PARAM_FINAL_5_MODEL.csv"
    )
    PARAM_FINAL_DICT = {
        r["Model"]: {
            "n_estimators": int(r["n_estimators"]),
            "bandwidth_k": int(r["bandwidth_k"]),
            "tweedie_variance_power": float(
                r["tweedie_variance_power"]
            )
        }
        for _, r in tabel.iterrows()
    }

DAFTAR_5 = PILIHAN_5_MODEL["Model"].tolist()

if not ZIP_PENELITIAN.exists():
    raise FileNotFoundError(
        "Upload HASIL_LENGKAP_ESF_TUNA.zip dahulu."
    )

# ------------------------------------------------
# 1. FUNGSI MEMBACA CSV DARI ZIP
# ------------------------------------------------

def baca_zip_csv(nama_file):
    with zipfile.ZipFile(ZIP_PENELITIAN, "r") as z:
        cocok = [
            n for n in z.namelist()
            if Path(n).name.lower() == nama_file.lower()
        ]

        if not cocok:
            raise FileNotFoundError(
                f"{nama_file} tidak ditemukan dalam ZIP."
            )

        with z.open(cocok[0]) as f:
            return pd.read_csv(f)

# ------------------------------------------------
# 2. GRID LENGKAP DAN GRID ABU
# ------------------------------------------------

GRID_LENGKAP = baca_zip_csv(
    "GRID_LENGKAP_WPP_572_573.csv"
)

# Normalisasi nama koordinat
rename = {}
for c in GRID_LENGKAP.columns:
    c_low = str(c).strip().lower()
    if c_low in ["longitude", "lon", "bujur"]:
        rename[c] = "Longitude"
    if c_low in ["latitude", "lat", "lintang"]:
        rename[c] = "Latitude"

GRID_LENGKAP = GRID_LENGKAP.rename(
    columns=rename
)

if not {"Longitude", "Latitude"}.issubset(
    GRID_LENGKAP.columns
):
    raise ValueError(
        "Grid lengkap harus mempunyai Longitude dan Latitude."
    )

GRID_LENGKAP = GRID_LENGKAP.drop_duplicates(
    subset=["Longitude", "Latitude"]
).reset_index(drop=True)

BASE = DATASET_MODEL["BASELINE"]

coord_obs_lonlat = BASE[
    ["Longitude", "Latitude"]
].to_numpy(float)

coord_grid_lonlat = GRID_LENGKAP[
    ["Longitude", "Latitude"]
].to_numpy(float)

# Identifikasi grid observasi dengan pusat koordinat asli.
# Tidak melakukan snapping/menggeser koordinat.
nn = NearestNeighbors(n_neighbors=1)
nn.fit(coord_obs_lonlat)

dist_derajat, _ = nn.kneighbors(
    coord_grid_lonlat
)

mask_abu = dist_derajat[:, 0] > 1e-5

GRID_ABU = GRID_LENGKAP.loc[
    mask_abu
].copy().reset_index(drop=True)

print("Jumlah observasi      :", len(BASE))
print("Jumlah grid lengkap   :", len(GRID_LENGKAP))
print("Jumlah grid abu-abu   :", len(GRID_ABU))

# ------------------------------------------------
# 3. MUAT 10 VARIABEL X UNTUK GRID ABU
# ------------------------------------------------

if FILE_GRID_DENGAN_X.exists():

    sumber_X = pd.read_csv(FILE_GRID_DENGAN_X)

    if not {"Longitude", "Latitude"}.issubset(
        sumber_X.columns
    ):
        raise ValueError(
            "File grid X harus memiliki Longitude, Latitude."
        )

    if sumber_X.duplicated(
        ["Longitude", "Latitude"]
    ).any():
        raise ValueError(
            "File grid X mengandung koordinat duplikat."
        )

    # Merge dengan toleransi koordinat asli:
    # pembulatan hanya untuk mencocokkan angka CSV,
    # tidak mengubah posisi titik pada peta.
    GRID_ABU["_lon_key"] = GRID_ABU["Longitude"].round(6)
    GRID_ABU["_lat_key"] = GRID_ABU["Latitude"].round(6)

    sumber_X["_lon_key"] = sumber_X["Longitude"].round(6)
    sumber_X["_lat_key"] = sumber_X["Latitude"].round(6)

    if sumber_X.duplicated(
        ["_lon_key", "_lat_key"]
    ).any():
        raise ValueError(
            "Ada koordinat yang menjadi duplikat saat pencocokan."
        )

    if not set(VAR_X).issubset(sumber_X.columns):
        raise ValueError(
            "File grid X belum memiliki 10 prediktor."
        )

    # Utamakan nilai dari file grid X yang diupload.
    GRID_ABU = GRID_ABU.drop(
        columns=[
            c for c in VAR_X
            if c in GRID_ABU.columns
        ]
    )

    GRID_ABU = GRID_ABU.merge(
        sumber_X[
            ["_lon_key", "_lat_key"] + VAR_X
        ],
        on=["_lon_key", "_lat_key"],
        how="left",
        validate="one_to_one"
    )

    GRID_ABU = GRID_ABU.drop(
        columns=["_lon_key", "_lat_key"]
    )

# ------------------------------------------------
# 4. JIKA X BELUM LENGKAP, BUAT TEMPLATE
# ------------------------------------------------

TEMPLATE_X = GRID_ABU[
    ["Longitude", "Latitude"]
].copy()

for c in VAR_X:
    TEMPLATE_X[c] = (
        pd.to_numeric(GRID_ABU[c], errors="coerce")
        if c in GRID_ABU.columns
        else np.nan
    )

TEMPLATE_X.to_csv(
    FOLDER_ANALISIS / "TEMPLATE_GRID_ABU_10_X.csv",
    index=False
)

jumlah_belum_lengkap = int(
    TEMPLATE_X[VAR_X].isna().any(axis=1).sum()
)

if jumlah_belum_lengkap > 0:
    print("\n⚠️ BELUM BISA MEMBUAT PETA PREDIKSI")
    print(
        "Jumlah grid abu-abu dengan X tidak lengkap:",
        jumlah_belum_lengkap
    )
    print(
        "Template disimpan di Google Drive:\n",
        FOLDER_ANALISIS / "TEMPLATE_GRID_ABU_10_X.csv"
    )
    print(
        "\nLengkapi 10 nilai oseanografi dengan data "
        "untuk lokasi dan periode yang sesuai, "
        "lalu upload sebagai:\n",
        FILE_GRID_DENGAN_X
    )

    raise RuntimeError(
        "Prediksi dihentikan karena prediktor grid "
        "abu-abu belum lengkap. Tidak ada nilai X "
        "atau CPUE yang diisi secara sembarang."
    )

for c in VAR_X:
    GRID_ABU[c] = pd.to_numeric(
        GRID_ABU[c], errors="coerce"
    )

if not np.isfinite(
    GRID_ABU[VAR_X].to_numpy(float)
).all():
    raise ValueError(
        "Grid X memiliki NaN atau infinity."
    )

print("\n✅ 10 variabel grid abu-abu lengkap.")

# ------------------------------------------------
# 5. HITUNG JARAK OBSERVASI DAN GRID
# ------------------------------------------------

coord_train = BASE[
    ["Latitude", "Longitude"]
].to_numpy(float)

coord_target = GRID_ABU[
    ["Latitude", "Longitude"]
].to_numpy(float)

D_train = haversine_distances(
    np.radians(coord_train)
) * R_BUMI_KM

np.fill_diagonal(D_train, np.inf)

D_cross = haversine_distances(
    np.radians(coord_target),
    np.radians(coord_train)
) * R_BUMI_KM

n_train = len(coord_train)
n_target = len(coord_target)

# ------------------------------------------------
# 6. MATRIKS W ORIGINAL DAN CROSS
# ------------------------------------------------

def W_train_asli(nama_w):

    if nama_w == "KNN5":
        W = np.zeros((n_train, n_train), dtype=float)
        idx = np.argpartition(
            D_train, kth=4, axis=1
        )[:, :5]
        W[np.arange(n_train)[:, None], idx] = 1.0
        W = np.maximum(W, W.T)

    elif nama_w == "DISTANCE":
        W = (D_train <= 122.33).astype(float)

    elif nama_w == "IDW":
        with np.errstate(divide="ignore", invalid="ignore"):
            W = np.where(
                (D_train <= 203.88) & (D_train > 0),
                1.0 / D_train,
                0.0
            )
    else:
        raise ValueError(f"W tidak dikenal: {nama_w}")

    np.fill_diagonal(W, 0.0)
    return W

def W_cross_baru(nama_w):

    if nama_w == "KNN5":
        # Aproksimasi: lima tetangga observasi
        # terdekat untuk setiap lokasi grid baru.
        W = np.zeros_like(D_cross)
        idx = np.argpartition(
            D_cross, kth=4, axis=1
        )[:, :5]
        W[np.arange(n_target)[:, None], idx] = 1.0

    elif nama_w == "DISTANCE":
        W = (D_cross <= 122.33).astype(float)

    elif nama_w == "IDW":
        with np.errstate(divide="ignore", invalid="ignore"):
            W = np.where(
                (D_cross <= 203.88) & (D_cross > 0),
                1.0 / D_cross,
                0.0
            )
    else:
        raise ValueError(f"W tidak dikenal: {nama_w}")

    return W

# Cache agar tidak menghitung W berulang
CACHE_EV = {}

# ------------------------------------------------
# 7. EKSTENSI EV KE LOKASI BARU
# ------------------------------------------------

def ekstensi_ev(kode, daftar_ev):

    if len(daftar_ev) == 0:
        return pd.DataFrame(index=GRID_ABU.index)

    nama_metode, nama_w = kode.split("_", 1)

    if nama_w not in CACHE_EV:
        W = W_train_asli(nama_w)

        M = np.eye(n_train) - np.ones(
            (n_train, n_train)
        ) / n_train

        C = M @ W @ M

        lam_semua = np.linalg.eigvalsh(C)[::-1]
        skala = np.max(np.abs(lam_semua))
        tol = 1e-10 * max(1.0, skala)

        lam_positif = lam_semua[lam_semua > tol]

        W_cross = W_cross_baru(nama_w)

        # Pemusatan cross terhadap referensi TRAIN
        C_cross = (
            W_cross
            - W_cross.mean(axis=1, keepdims=True)
            - W.mean(axis=0, keepdims=True)
            + W.mean()
        )

        CACHE_EV[nama_w] = (
            C,
            lam_positif,
            C_cross
        )

    C, lam_positif, C_cross = CACHE_EV[nama_w]

    # Nama EV harus berbentuk EV1, EV2, ...
    posisi_ev = []

    for c in daftar_ev:
        match = re.fullmatch(r"EV(\d+)", str(c))

        if match is None:
            raise ValueError(
                f"Nama {c} tidak sesuai EV1, EV2, ..."
            )

        posisi = int(match.group(1)) - 1

        if posisi < 0 or posisi >= len(lam_positif):
            raise ValueError(
                f"Indeks eigenvector tidak valid: {c}"
            )

        posisi_ev.append(posisi)

    lambdas = lam_positif[posisi_ev]

    E_obs = DATASET_MODEL[
        kode
    ][daftar_ev].to_numpy(float)

    # Verifikasi EV merupakan eigenvector dari
    # matriks MWM TRAIN, bukan kolom lain.
    resid_eigen = C @ E_obs - E_obs * lambdas[None, :]

    relatif = (
        np.linalg.norm(resid_eigen, axis=0)
        /
        np.maximum(
            np.linalg.norm(E_obs, axis=0) *
            np.abs(lambdas),
            1e-12
        )
    )

    if np.max(relatif) > 1e-4:
        raise ValueError(
            f"Eigenpair {kode} tidak cocok dengan W. "
            f"Galat relatif maksimum = {np.max(relatif):.4g}. "
            "Periksa urutan EV dan definisi W sebelum prediksi."
        )

    # Ekstensi aproksimasi eigenvector
    E_new = (C_cross @ E_obs) / lambdas[None, :]

    if not np.isfinite(E_new).all():
        raise ValueError(f"Ekstensi EV tidak valid: {kode}")

    return pd.DataFrame(
        E_new,
        columns=daftar_ev,
        index=GRID_ABU.index
    )

# ------------------------------------------------
# 8. PREDIKSI MODEL LOKAL DENGAN CHECKPOINT
# ------------------------------------------------

SEMUA_PETA = []

for kode in DAFTAR_5:

    print("\nMEMBUAT PETA:", kode)

    train_df = DATASET_MODEL[kode]

    fitur = [
        c for c in train_df.columns
        if c not in ["Longitude", "Latitude", "cpue"]
    ]

    ev_cols = [
        c for c in fitur if c not in VAR_X
    ]

    X_train = train_df[fitur].to_numpy(float)
    y_train = train_df["cpue"].to_numpy(float)

    X_target_df = GRID_ABU[VAR_X].copy()

    if kode != "BASELINE":
        X_target_df = pd.concat(
            [
                X_target_df,
                ekstensi_ev(kode, ev_cols)
            ],
            axis=1
        )

    X_target = X_target_df[fitur].to_numpy(float)

    if not np.isfinite(X_target).all():
        raise ValueError(
            f"Fitur grid tidak valid: {kode}"
        )

    konfigurasi = PARAM_FINAL_DICT[kode]
    k = int(konfigurasi["bandwidth_k"])

    params = {
        **PARAM_TETAP,
        "n_estimators": int(
            konfigurasi["n_estimators"]
        ),
        "tweedie_variance_power": float(
            konfigurasi["tweedie_variance_power"]
        ),
        "objective": "reg:tweedie",
        "tree_method": "hist",
        "random_state": SEED,
        "n_jobs": 1,
        "verbosity": 0
    }

    # Identitas input untuk validasi checkpoint
    h = hashlib.sha256()

    for arr in [X_train, y_train, X_target, D_cross]:
        h.update(
            np.ascontiguousarray(
                arr, dtype=np.float64
            ).tobytes()
        )

    h.update(
        json.dumps({
            "model": kode,
            "params": konfigurasi,
            "params_tetap": PARAM_TETAP,
            "version": "EV_extension_approx_v1"
        }, sort_keys=True).encode()
    )

    id_input = h.hexdigest()[:20]

    path_pred = (
        FOLDER_ANALISIS /
        f"CHECKPOINT_GRID_{kode}.csv"
    )
    path_meta = (
        FOLDER_ANALISIS /
        f"CHECKPOINT_GRID_{kode}.json"
    )

    # Persiapan checkpoint
    progress = pd.DataFrame({
        "Index": np.arange(n_target),
        "CPUE_Prediksi": np.nan
    })

    if path_pred.exists() and path_meta.exists():
        with open(path_meta, "r") as f:
            metadata_lama = json.load(f)

        lama = pd.read_csv(path_pred)

        if (
            metadata_lama.get("ID_Input") == id_input
            and len(lama) == n_target
            and np.array_equal(
                lama["Index"].to_numpy(dtype=int),
                np.arange(n_target)
            )
        ):
            progress = lama
            print("✅ Checkpoint kompatibel ditemukan.")

    # Simpan identitas input baru
    with open(path_meta, "w") as f:
        json.dump(
            {"Model": kode, "ID_Input": id_input},
            f, indent=2
        )

    belum = np.flatnonzero(
        progress["CPUE_Prediksi"].isna()
    )

    for no, i in enumerate(belum, start=1):

        d = D_cross[i]

        b = max(
            float(np.partition(d, k - 1)[k - 1]),
            1e-6
        )

        bobot = np.exp(-0.5 * (d / b) ** 2)
        bobot /= bobot.mean()

        model = XGBRegressor(**params)

        model.fit(
            X_train,
            y_train,
            sample_weight=bobot
        )

        yhat = float(
            model.predict(
                X_target[i:i+1]
            )[0]
        )

        if not np.isfinite(yhat):
            raise ValueError(
                f"Prediksi tidak valid: {kode}, titik {i}"
            )

        progress.loc[i, "CPUE_Prediksi"] = yhat

        if no % 25 == 0 or no == len(belum):
            progress.to_csv(path_pred, index=False)
            print(
                f"  {progress['CPUE_Prediksi'].notna().sum()}"
                f"/{n_target} titik selesai"
            )

    hasil = GRID_ABU[
        ["Longitude", "Latitude"]
    ].copy()

    hasil["Model"] = kode
    hasil["CPUE_Prediksi"] = (
        progress["CPUE_Prediksi"].to_numpy(float)
    )

    hasil.to_csv(
        FOLDER_ANALISIS /
        f"PREDIKSI_GRID_ABU_{kode}.csv",
        index=False
    )

    SEMUA_PETA.append(hasil)

# ------------------------------------------------
# 9. GABUNGKAN PREDIKSI
# ------------------------------------------------

PREDIKSI_GRID_5_MODEL = pd.concat(
    SEMUA_PETA,
    ignore_index=True
)

PREDIKSI_GRID_5_MODEL.to_csv(
    FOLDER_ANALISIS / "PREDIKSI_GRID_ABU_5_MODEL.csv",
    index=False
)

# Skala warna bersama agar adil
vmin = float(
    PREDIKSI_GRID_5_MODEL["CPUE_Prediksi"].min()
)
vmax = float(
    PREDIKSI_GRID_5_MODEL["CPUE_Prediksi"].max()
)

if vmax <= vmin:
    vmax = vmin + 1e-8

# ------------------------------------------------
# 10. LIMA PETA TERPISAH
# ------------------------------------------------

for hasil in SEMUA_PETA:

    kode = hasil["Model"].iloc[0]

    fig, ax = plt.subplots(
        figsize=(11, 6),
        dpi=140
    )

    ax.scatter(
        BASE["Longitude"],
        BASE["Latitude"],
        color="#aec7e8",
        s=12,
        alpha=0.5,
        label="Lokasi pengamatan"
    )

    sc = ax.scatter(
        hasil["Longitude"],
        hasil["Latitude"],
        c=hasil["CPUE_Prediksi"],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        marker="s",
        s=50,
        linewidths=0
    )

    fig.colorbar(
        sc, ax=ax, label="Prediksi CPUE"
    )

    ax.set_title(
        f"Peta Prediksi CPUE - {kode}\n"
        "Grid Belum Diamati WPP 572-573"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.15)
    ax.legend(loc="best")

    plt.tight_layout()
    plt.savefig(
        FOLDER_ANALISIS /
        f"PETA_PREDIKSI_CPUE_{kode}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# ------------------------------------------------
# 11. GRAFIK GABUNGAN LIMA PETA
# ------------------------------------------------

fig, axes = plt.subplots(
    3, 2,
    figsize=(16, 16),
    dpi=140
)

for ax, hasil in zip(
    axes.ravel(), SEMUA_PETA
):
    kode = hasil["Model"].iloc[0]

    sc = ax.scatter(
        hasil["Longitude"],
        hasil["Latitude"],
        c=hasil["CPUE_Prediksi"],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        marker="s",
        s=20,
        linewidths=0
    )

    ax.set_title(kode)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.15)

axes.ravel()[-1].axis("off")

fig.colorbar(
    sc,
    ax=axes.ravel().tolist(),
    label="Prediksi CPUE",
    shrink=0.62
)

fig.suptitle(
    "Lima Peta Prediksi CPUE\n"
    "Baseline dan Empat Metode ESF Terbaik",
    fontsize=15
)

fig.subplots_adjust(
    top=0.92,
    right=0.88,
    hspace=0.3,
    wspace=0.3
)

plt.savefig(
    FOLDER_ANALISIS /
    "GABUNGAN_5_PETA_PREDIKSI_CPUE.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------
# 12. ARSIP RINGKAS HASIL ANALISIS
# ------------------------------------------------

import shutil

PATH_ZIP = shutil.make_archive(
    "/content/ANALISIS_AKHIR_5_MODEL",
    "zip",
    str(FOLDER_ANALISIS)
)

print("\n✅ CELL 3 SELESAI")
print("✅ 5 model memprediksi grid abu-abu")
print("✅ 5 peta dengan skala warna sama")
print("✅ CSV, peta, checkpoint tersimpan")
print("✅ ZIP tersedia:", PATH_ZIP)
print("✅ Folder Google Drive:", FOLDER_ANALISIS)
